# DMP-WCEBleedNet for WCEBleedGen
- Segmentation + Classification of bleeding in WCE images
- Includes metrics, confusion matrices, ROC, learning curves


Cell 2: Imports

In [1]:
# -------------------------
# Imports
# -------------------------
import os
import random
import shutil
from tqdm import tqdm
from PIL import Image
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns


Cell 3: Dataset Split Function

In [2]:
# -------------------------
# Dataset Split Function
# -------------------------
def split_dataset(data_dir, output_dir, seed=42):
    random.seed(seed)
    os.makedirs(output_dir, exist_ok=True)
    pairs = []

    # Bleeding samples
    bleed_img_dir = os.path.join(data_dir, "bleeding", "Images")
    bleed_mask_dir = os.path.join(data_dir, "bleeding", "Annotations")
    for f in os.listdir(bleed_img_dir):
        if f.endswith(".png"):
            mask_path = os.path.join(bleed_mask_dir, f.replace("img-", "ann-"))
            if os.path.exists(mask_path):
                pairs.append((os.path.join(bleed_img_dir, f), mask_path))

    # Non-bleeding samples
    non_img_dir = os.path.join(data_dir, "non-bleeding", "images")
    non_mask_dir = os.path.join(data_dir, "non-bleeding", "annotation")
    for f in os.listdir(non_img_dir):
        if f.endswith(".png"):
            mask_path = os.path.join(non_mask_dir, f.replace("img-", "ann-"))
            if os.path.exists(mask_path):
                pairs.append((os.path.join(non_img_dir, f), mask_path))

    # Split
    random.shuffle(pairs)
    n = len(pairs)
    n_train = int(0.7*n)
    n_val = int(0.2*n)
    train = pairs[:n_train]
    val = pairs[n_train:n_train+n_val]
    test = pairs[n_train+n_val:]

    for split, items in [("train",train),("val",val),("test",test)]:
        img_out = os.path.join(output_dir, split, "images")
        mask_out = os.path.join(output_dir, split, "masks")
        os.makedirs(img_out, exist_ok=True)
        os.makedirs(mask_out, exist_ok=True)
        for img_src, mask_src in items:
            shutil.copy(img_src, os.path.join(img_out, os.path.basename(img_src)))
            shutil.copy(mask_src, os.path.join(mask_out, os.path.basename(mask_src)))

    print(f"Split done. train={len(train)} val={len(val)} test={len(test)}")


Cell 4: Dataset Loader

In [3]:
# -------------------------
# Dataset Loader
# -------------------------
class WCEBleedDataset(Dataset):
    def __init__(self, images_dir, masks_dir, transform=None, img_size=128):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.files = sorted([f for f in os.listdir(images_dir) if f.endswith(".png")])
        self.transform = transform
        self.img_size = img_size

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        img_path = os.path.join(self.images_dir, fname)
        mask_path = os.path.join(self.masks_dir, fname.replace("img-", "ann-"))

        img = np.array(Image.open(img_path).convert("RGB").resize((self.img_size,self.img_size)))
        mask = np.array(Image.open(mask_path).convert("L").resize((self.img_size,self.img_size)))
        mask = (mask > 127).astype(np.uint8)

        cls_label = int(mask.sum() > 0)

        if self.transform:
            transformed = self.transform(image=img, mask=mask)
            img_t = transformed["image"]
            mask_t = transformed["mask"].unsqueeze(0).float()
        else:
            img_t = torch.from_numpy(img).permute(2,0,1).float()/255.0
            mask_t = torch.from_numpy(mask).unsqueeze(0).float()

        return img_t, mask_t, torch.tensor(cls_label, dtype=torch.long), fname


Cell 5: Model Definition

In [4]:
# -------------------------
# Model: DMPBlock + DMP_WCEBleedNet
# -------------------------
class DMPBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        mid = out_ch // 2
        self.branch1 = nn.Sequential(nn.Conv2d(in_ch, mid, 1), nn.BatchNorm2d(mid), nn.ReLU())
        self.branch3 = nn.Sequential(nn.Conv2d(in_ch, mid, 3, padding=1), nn.BatchNorm2d(mid), nn.ReLU())
        self.branch5 = nn.Sequential(nn.Conv2d(in_ch, mid, 5, padding=2), nn.BatchNorm2d(mid), nn.ReLU())
        self.fuse = nn.Sequential(nn.Conv2d(mid*3, out_ch, 1), nn.BatchNorm2d(out_ch), nn.ReLU())

    def forward(self, x):
        return self.fuse(torch.cat([self.branch1(x), self.branch3(x), self.branch5(x)], dim=1))

class DMP_WCEBleedNet(nn.Module):
    def __init__(self, in_channels=3, base_filters=16, num_classes=2):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(in_channels, base_filters, 3, padding=1), nn.BatchNorm2d(base_filters), nn.ReLU(), DMPBlock(base_filters, base_filters))
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = nn.Sequential(nn.Conv2d(base_filters, base_filters*2, 3, padding=1), nn.BatchNorm2d(base_filters*2), nn.ReLU(), DMPBlock(base_filters*2, base_filters*2))
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = nn.Sequential(nn.Conv2d(base_filters*2, base_filters*4, 3, padding=1), nn.BatchNorm2d(base_filters*4), nn.ReLU(), DMPBlock(base_filters*4, base_filters*4))
        self.bottleneck = nn.Sequential(nn.MaxPool2d(2), nn.Conv2d(base_filters*4, base_filters*8, 3, padding=1), nn.BatchNorm2d(base_filters*8), nn.ReLU())

        self.up3 = nn.ConvTranspose2d(base_filters*8, base_filters*4, 2, 2)
        self.dec3 = nn.Conv2d(base_filters*8, base_filters*4, 3, padding=1)
        self.up2 = nn.ConvTranspose2d(base_filters*4, base_filters*2, 2, 2)
        self.dec2 = nn.Conv2d(base_filters*4, base_filters*2, 3, padding=1)
        self.up1 = nn.ConvTranspose2d(base_filters*2, base_filters, 2, 2)
        self.dec1 = nn.Conv2d(base_filters*2, base_filters, 3, padding=1)
        self.seg_out = nn.Conv2d(base_filters, 1, 1)

        self.classifier = nn.Sequential(nn.AdaptiveAvgPool2d((1,1)), nn.Flatten(), nn.Linear(base_filters*8, 64), nn.ReLU(), nn.Linear(64, num_classes))

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b = self.bottleneck(e3)
        cls_logits = self.classifier(b)
        d3 = self.up3(b); d3 = torch.cat([d3,e3],1); d3 = F.relu(self.dec3(d3))
        d2 = self.up2(d3); d2 = torch.cat([d2,e2],1); d2 = F.relu(self.dec2(d2))
        d1 = self.up1(d2); d1 = torch.cat([d1,e1],1); d1 = F.relu(self.dec1(d1))
        seg_logits = self.seg_out(d1)
        return seg_logits, cls_logits


Cell 6: Metrics Functions

In [5]:
# -------------------------
# Metrics Functions
# -------------------------
def dice_torch(pred, target, eps=1e-6):
    pred = pred.view(-1); target = target.view(-1)
    inter = (pred * target).sum()
    return (2*inter + eps) / (pred.sum() + target.sum() + eps)

def iou_torch(pred, target, eps=1e-6):
    pred = pred.view(-1); target = target.view(-1)
    inter = (pred * target).sum()
    union = (pred + target).clamp(0,1).sum()
    return (inter + eps) / (union + eps)

def train_epoch(model, device, loader, opt, loss_seg, loss_cls):
    model.train()
    total_loss = 0
    for imgs, masks, cls, _ in tqdm(loader, desc='Train'):
        imgs, masks, cls = imgs.to(device), masks.to(device), cls.to(device)
        opt.zero_grad()
        seg_logits, cls_logits = model(imgs)
        loss = loss_seg(seg_logits, masks) + loss_cls(cls_logits, cls)
        loss.backward()
        opt.step()
        total_loss += loss.item() * imgs.size(0)
    return total_loss / len(loader.dataset)

def eval_epoch(model, device, loader, loss_seg, loss_cls):
    model.eval()
    y_true, y_pred = [], []
    dice_scores, iou_scores = [], []
    total_loss = 0
    y_true_masks, y_pred_masks = [], []
    with torch.no_grad():
        for imgs, masks, cls, _ in tqdm(loader, desc='Val/Test'):
            imgs, masks, cls = imgs.to(device), masks.to(device), cls.to(device)
            seg_logits, cls_logits = model(imgs)
            loss = loss_seg(seg_logits, masks) + loss_cls(cls_logits, cls)
            total_loss += loss.item() * imgs.size(0)

            pred_mask = (torch.sigmoid(seg_logits) > 0.5).float()
            y_true_masks.extend(masks.cpu().numpy())
            y_pred_masks.extend(pred_mask.cpu().numpy())

            for pm, gt in zip(pred_mask, masks):
                dice_scores.append(dice_torch(pm, gt).item())
                iou_scores.append(iou_torch(pm, gt).item())

            preds = torch.argmax(cls_logits, 1).cpu().numpy()
            y_pred.extend(preds.tolist())
            y_true.extend(cls.cpu().numpy().tolist())

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)
    return {
        "loss": total_loss/len(loader.dataset),
        "dice": np.mean(dice_scores),
        "iou": np.mean(iou_scores),
        "acc": acc,
        "prec": prec,
        "rec": rec,
        "f1": f1,
        "cm": cm,
        "y_true_masks": y_true_masks,
        "y_pred_masks": y_pred_masks,
        "y_true_cls": y_true,
        "y_pred_cls": y_pred
    }


Cell 7: Plotting Functions

In [6]:
# -------------------------
# Plotting Utilities
# -------------------------
def plot_confusion_matrix(cm, classes, title, save_path):
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

def plot_segmentation_confusion_matrix(y_true_masks, y_pred_masks, save_path):
    y_true = np.concatenate([m.flatten() for m in y_true_masks])
    y_pred = np.concatenate([m.flatten() for m in y_pred_masks])
    cm = confusion_matrix(y_true, y_pred)
    plot_confusion_matrix(cm, ['Non-Bleed', 'Bleed'], 'Segmentation Confusion Matrix', save_path)

def plot_roc_curve(y_true, y_scores, save_path):
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(6,5))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
    plt.plot([0,1], [0,1], color='navy', lw=2, linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Classification ROC Curve')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

def plot_learning_curves(train_losses, val_losses, val_dice, save_dir):
    epochs = range(1, len(train_losses)+1)
    plt.figure()
    plt.plot(epochs, train_losses, 'b-', label='Train Loss')
    plt.plot(epochs, val_losses, 'r-', label='Val Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('Learning Curve')
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'learning_curve.png'))
    plt.close()

    plt.figure()
    plt.plot(epochs, val_dice, 'g-', label='Val Dice')
    plt.xlabel('Epochs')
    plt.ylabel('Dice')
    plt.title('Validation Dice Curve')
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'dice_curve.png'))
    plt.close()


Cell 8: Dataset Setup

In [7]:
# -------------------------
# Dataset Setup
# -------------------------
DATA_DIR = 'C:/Users/aditi/OneDrive/Desktop/WCE Main/WCE Main/WCEBleedGen' #path
OUTPUT_DIR = 'dataset_split'
IMG_SIZE = 128
BATCH_SIZE = 4

split_dataset(DATA_DIR, OUTPUT_DIR)

transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2()
])

train_ds = WCEBleedDataset(os.path.join(OUTPUT_DIR,'train','images'), os.path.join(OUTPUT_DIR,'train','masks'), transform, IMG_SIZE)
val_ds = WCEBleedDataset(os.path.join(OUTPUT_DIR,'val','images'), os.path.join(OUTPUT_DIR,'val','masks'), transform, IMG_SIZE)
test_ds = WCEBleedDataset(os.path.join(OUTPUT_DIR,'test','images'), os.path.join(OUTPUT_DIR,'test','masks'), transform, IMG_SIZE)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)


Split done. train=1832 val=523 test=263


Cell 9: Model & Optimizer Setup

In [8]:
# -------------------------
# Model & Optimizer
# -------------------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = DMP_WCEBleedNet().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_seg = nn.BCEWithLogitsLoss()
loss_cls = nn.CrossEntropyLoss()


Cell 10: Training Loop & Plots

In [ ]:
import csv

# -------------------------
# Training Loop with Metrics, Plotting & CSV Logging
# -------------------------

EPOCHS = 20
train_losses, val_losses, val_dice = [], [], []

# Prepare CSV file
csv_file = os.path.join(OUTPUT_DIR, "training_metrics.csv")
with open(csv_file, mode="w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "epoch", "train_loss", "val_loss", 
        "seg_dice", "seg_iou", 
        "cls_acc", "cls_prec", "cls_rec", "cls_f1"
    ])

for epoch in range(EPOCHS):
    # 1️⃣ Train for one epoch
    train_loss = train_epoch(model, device, train_loader, opt, loss_seg, loss_cls)
    
    # 2️⃣ Evaluate on validation set
    val_metrics = eval_epoch(model, device, val_loader, loss_seg, loss_cls)
    
    # 3️⃣ Save metrics for plotting
    train_losses.append(train_loss)
    val_losses.append(val_metrics['loss'])
    val_dice.append(val_metrics['dice'])
    
    # 4️⃣ Print metrics clearly
    print(f"\n=== Epoch {epoch+1}/{EPOCHS} ===")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_metrics['loss']:.4f}")
    print(f"Seg Dice: {val_metrics['dice']:.4f} | IOU: {val_metrics['iou']:.4f}")
    print(f"Cls Acc: {val_metrics['acc']:.4f} | Prec: {val_metrics['prec']:.4f} | Rec: {val_metrics['rec']:.4f} | F1: {val_metrics['f1']:.4f}")
    
    # 5️⃣ Append metrics to CSV
    with open(csv_file, mode="a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            epoch+1, train_loss, val_metrics['loss'],
            val_metrics['dice'], val_metrics['iou'],
            val_metrics['acc'], val_metrics['prec'], val_metrics['rec'], val_metrics['f1']
        ])

# -------------------------
# Plot Confusion Matrices
# -------------------------
plot_confusion_matrix(
    val_metrics['cm'],
    ['Non-Bleed','Bleed'],
    'Classification Confusion Matrix',
    os.path.join(OUTPUT_DIR,'cls_confusion_matrix.png')
)

plot_segmentation_confusion_matrix(
    val_metrics['y_true_masks'],
    val_metrics['y_pred_masks'],
    os.path.join(OUTPUT_DIR,'seg_confusion_matrix.png')
)

# -------------------------
# Plot ROC Curve
# -------------------------
y_scores = []
with torch.no_grad():
    for imgs, masks, cls, _ in val_loader:
        imgs = imgs.to(device)
        probs = torch.softmax(model(imgs)[1], dim=1)[:,1].cpu().numpy()  # probability of class 1 (bleeding)
        y_scores.extend(probs)
        
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# The true and predicted values are already prepared by the lines above this
y_true_labels = np.array(val_metrics['y_true_cls'])
y_predicted_probs = np.array(y_scores)

# Calculate the metrics
mae = mean_absolute_error(y_true_labels, y_predicted_probs)
mse = mean_squared_error(y_true_labels, y_predicted_probs)
r2 = r2_score(y_true_labels, y_predicted_probs)

# Print the results in a clear format
print("--- Regression-Style Error Metrics for DMP Model ---")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Mean Squared Error (MSE):  {mse:.4f}")
print(f"R-squared (R2):          {r2:.4f}")
print("----------------------------------------------------")
plot_roc_curve(
    np.array(val_metrics['y_true_cls']),
    np.array(y_scores),
    os.path.join(OUTPUT_DIR,'roc_curve.png')
)

# -------------------------
# Plot Learning & Dice Curves
# -------------------------
plot_learning_curves(train_losses, val_losses, val_dice, OUTPUT_DIR)



Val/Test: 100%|██████████| 119/119 [00:04<00:00, 29.23it/s]



=== Epoch 1/20 ===
Train Loss: 0.8470
Val Loss: 0.5595
Seg Dice: 0.6018 | IOU: 0.5635
Cls Acc: 0.8734 | Prec: 0.9333 | Rec: 0.8099 | F1: 0.8673


Val/Test: 100%|██████████| 119/119 [00:04<00:00, 27.94it/s]



=== Epoch 2/20 ===
Train Loss: 0.6569
Val Loss: 0.4675
Seg Dice: 0.6941 | IOU: 0.6401
Cls Acc: 0.8924 | Prec: 0.9569 | Rec: 0.8264 | F1: 0.8869


Val/Test: 100%|██████████| 119/119 [00:04<00:00, 29.06it/s]


...


=== Epoch 18/20 ===
Train Loss: 0.3052
Val Loss: 0.2252
Seg Dice: 0.6946 | IOU: 0.6398
Cls Acc: 0.9747 | Prec: 0.9832 | Rec: 0.9669 | F1: 0.9750


Val/Test: 100%|██████████| 119/119 [00:04<00:00, 27.16it/s]



=== Epoch 19/20 ===
Train Loss: 0.2665
Val Loss: 0.2530
Seg Dice: 0.7103 | IOU: 0.6566
Cls Acc: 0.9620 | Prec: 0.9667 | Rec: 0.9587 | F1: 0.9627


Val/Test: 100%|██████████| 119/119 [00:04<00:00, 28.89it/s]



=== Epoch 20/20 ===
Train Loss: 0.2873
Val Loss: 0.2691
Seg Dice: 0.6854 | IOU: 0.6345
Cls Acc: 0.9620 | Prec: 0.9375 | Rec: 0.9917 | F1: 0.9639
